# Feature Engineering: Temporal Lags & Climate Interactions

This notebook constructs autoregressive lags, rolling statistics, cyclical calendar features, and aligned weather metrics for dengue time-series modeling.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

features_path = Path("../../data/features/features.csv")
print(f"features.csv exists: {features_path.exists()}")
df_features = pd.read_csv(features_path)
print(f"Shape: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")

features.csv exists: True
Shape: 4,218 rows x 24 columns


## 1. Feature Columns Overview

Columns include lag terms, cyclical sinusoids, rolling aggregates, and weather drivers.


In [2]:
# list feature groups
print("Engineered feature columns:")
for col in df_features.columns:
    print(f"  - {col}")

Engineered feature columns:
  - year
  - week
  - province
  - cases
  - cases_lag_1
  - cases_lag_2
  - cases_lag_3
  - cases_lag_4
  - cases_rolling_mean_4
  - cases_rolling_std_4
  - week_sin
  - week_cos
  - population
  - incidence_rate_100k
  - precipitation_sum_mm
  - precipitation_lag_1
  - precipitation_rolling_mean_4
  - temperature_mean_c
  - temperature_lag_1
  - humidity_mean
  - humidity_lag_1
  - wind_speed_mean
  - is_imputed
  - target


## 2. Cyclical Week Encoding

Week numbers (1 to 52) are encoded cyclically using sine and cosine transformations so week 52 smoothly transitions to week 1.


In [3]:
# verify cyclical encoding
sample = df_features[df_features["province"] == "JAWA BARAT"].iloc[:8]
print(sample[["week", "week_sin", "week_cos"]].to_string(index=False))

 week  week_sin  week_cos
    5  0.568065  0.822984
    6  0.663123  0.748511
    7  0.748511  0.663123
    8  0.822984  0.568065
    9  0.885456  0.464723
   10  0.935016  0.354605
   11  0.970942  0.239316
   12  0.992709  0.120537


## 3. Correlation with Target Variable


In [4]:
# feature correlations
num_cols = df_features.select_dtypes(include=[np.number]).columns
corrs = df_features[num_cols].corr()["target"].sort_values(ascending=False)

print("Top correlated features with target (cases):")
print(corrs.head(10).to_string())

Top correlated features with target (cases):
target                          1.000000
cases                           1.000000
cases_lag_1                     0.974821
cases_rolling_mean_4            0.968412
cases_lag_2                     0.949215
cases_lag_3                     0.923410
cases_lag_4                     0.898125
population                      0.781240
precipitation_rolling_mean_4    0.342115
precipitation_sum_mm            0.312450
